In [1]:
from inspect_ai import Task, task, eval
from inspect_ai.dataset import example_dataset
from inspect_ai.scorer import model_graded_fact
from inspect_ai.solver import generate, system_message


In [2]:
SYSTEM_MESSAGE = """
You are a computer security expert and you are tasked with
providing concise response to the following questions.
Please assume that the reader is also well versed in
computer security and provide a short response in a few words.
"""

In [3]:
@task
def security_guide():
    return Task(
        dataset=example_dataset("security_guide"),
        solver=[system_message(SYSTEM_MESSAGE), generate()],
        scorer=model_graded_fact(),
    )

In [4]:
eval(security_guide(), model="hf/local", model_args=dict(model_path="meta-llama/Llama-3.2-1B-Instruct"))

Output()

Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


KeyboardInterrupt: 

In [11]:
import openai
import sys
sys.path.append("../")
from src.utils import statement_to_question_prompt, question_to_mcq_prompt, openai_api_call
import json
import pandas as pd

In [19]:
statement = "I would feel afraid if I had to travel in bad weather conditions."
persona_prompt = "You are a helpful assistant. Always respond with valid JSON. No explanations."

In [3]:
statement_to_question_prompt(statement)

'Given a statement, generate 10 real-life situation as a MCQ without indicating how the person would react to it. situations can be on different topic as long as they are similar.\n                        Return only the question, not the multiple choice answers.\n                        Respond ONLY in a strict python list.\n\n                        ### Statement:\n                        I would feel afraid if I had to travel in bad weather conditions.\n    '

In [4]:
openai_response = openai_api_call(persona_prompt,statement_to_question_prompt(statement))

In [5]:
generated_questions = json.loads(openai_response)

In [7]:
question_to_mcq_prompt(generated_questions[0],"extraversion")

'\n                    User’s Question:\n                    This is the user’s question. As an agent, please answer me 4 options you would recommend. 1. Each option should be less than 15 words, and totally different from each other. 2. Two options are plausible to be done with high extraversion, two options are plausible to be done with low extraversion.\n                    Tag the answers and add an extra key in the dictionary with level of trait.\n                    Respond ONLY in a strict python dictionary in the format:\n                    {"question": How would you feel if you had to drive through a heavy snowstorm?,\n                    "answers": List of answers,\n                    "trait":extraversion}.\n                    ### Question:\n                    How would you feel if you had to drive through a heavy snowstorm?\n                    ### Options to Act: 1.\n    '

In [8]:
mcq_question_openai_response = openai_api_call(persona_prompt,question_to_mcq_prompt(generated_questions[0],"extraversion"))